# Spatial teleconnection fingerprints in basin SST anomalies (GP regression)

We fit a spatially varying coefficient model:

\[
Y(t,s)=\mu(s)+\beta_N(s)N_t+\beta_D(s)D_t+\beta_{ND}(s)N_tD_t+\epsilon(t,s)
\]

where \(Y\) is the SST anomaly field in a given basin and \(N_t\), \(D_t\) are standardized RONI and DMI.
We then run counterfactual “mode experiments” by setting \((N,D)\) to fixed values (e.g., ±1σ, ±2σ).


In [1]:
import os, glob
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pymc as pm
import pytensor.tensor as pt


ERROR 1: PROJ: proj_create_from_database: Open of /home/k16v981/.conda/envs/my_env/share/proj failed


In [12]:
[d for d in dir(pm) if d[0].isupper()]

['ADVI',
 'AR',
 'ASVGD',
 'Approximation',
 'AsymmetricLaplace',
 'Bernoulli',
 'Beta',
 'BetaBinomial',
 'BinaryGibbsMetropolis',
 'BinaryMetropolis',
 'Binomial',
 'Bound',
 'CAR',
 'CallableTensor',
 'Categorical',
 'CategoricalGibbsMetropolis',
 'Cauchy',
 'CauchyProposal',
 'Censored',
 'ChiSquared',
 'CompoundStep',
 'Constant',
 'ConstantData',
 'Continuous',
 'CustomDist',
 'DEMetropolis',
 'DEMetropolisZ',
 'Data',
 'DensityDist',
 'Deterministic',
 'DictToArrayBijection',
 'DiracDelta',
 'Dirichlet',
 'DirichletMultinomial',
 'Discrete',
 'DiscreteUniform',
 'DiscreteWeibull',
 'Distribution',
 'ELPDData',
 'Empirical',
 'EulerMaruyama',
 'ExGaussian',
 'Exponential',
 'Flat',
 'FullRank',
 'FullRankADVI',
 'GARCH11',
 'Gamma',
 'GaussianRandomWalk',
 'GeneratorAdapter',
 'Geometric',
 'Group',
 'Gumbel',
 'HalfCauchy',
 'HalfFlat',
 'HalfNormal',
 'HalfStudentT',
 'HamiltonianMC',
 'HyperGeometric',
 'ImplicitGradient',
 'ImputationWarning',
 'IncorrectArgumentsError',
 'In

In [3]:
# -----------------------
# Config
# -----------------------
NETID = "k16v981"

BASIN = "arabian_gulf"   # change per run: arabian_gulf, gulf_oman, red_and_aden

BASIN_NC = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/basin_anoms/era5_sst_anom_{BASIN}_1950_2025.nc"
IDX_CSV  = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/roni_dmi_monthly_1950_2025.csv"

OUT_IDATA = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/gp_{BASIN}_roni_dmi_idata.nc"

RANDOM_SEED = 42

# Mode experiments in standardized units (since we z-score indices)
EXPERIMENTS = [
    {"name": "neutral (0,0)",                "N":  0.0, "D":  0.0},
    {"name": "El Niño (+1,0)",               "N": +1.0, "D":  0.0},
    {"name": "La Niña (-1,0)",               "N": -1.0, "D":  0.0},
    {"name": "pIOD (0,+1)",                  "N":  0.0, "D": +1.0},
    {"name": "nIOD (0,-1)",                  "N":  0.0, "D": -1.0},
    {"name": "Joint + (+1,+1)",              "N": +1.0, "D": +1.0},
    {"name": "Opposing El Niño (+1,-1)",     "N": +1.0, "D": -1.0},
    {"name": "Joint - (-1,-1)",              "N": -1.0, "D": -1.0},
    {"name": "Opposing La Niña (-1,+1)",     "N": -1.0, "D": +1.0},
    {"name": "Strong Joint - (-2,-2)",       "N": -2.0, "D": -2.0},
]


In [4]:
ds = xr.open_dataset(BASIN_NC)
da = ds["sst_anom"]  # (time, lat, lon)

lat_name = "latitude" if "latitude" in da.coords else "lat"
lon_name = "longitude" if "longitude" in da.coords else "lon"

# shift lon to [-180, 180] if needed for mapping
if float(da[lon_name].max()) > 180:
    lon = da[lon_name]
    lon_new = ((lon + 180) % 360) - 180
    da = da.assign_coords({lon_name: lon_new}).sortby(lon_name)

# stack spatial dims
da_st = da.stack(space=(lat_name, lon_name))  # (time, space)

# keep only wet basin points: non-NaN at first time
valid_space = np.isfinite(da_st.isel(time=0).values)
da_st = da_st.isel(space=valid_space)

Y = da_st.values.astype("float32")  # (T, S)
time = pd.to_datetime(da_st["time"].values)

# spatial coords list (S,)
space_index = da_st["space"].to_index()
lats = np.array([x[0] for x in space_index], dtype="float32")
lons = np.array([x[1] for x in space_index], dtype="float32")

X_space = np.column_stack([lons, lats]).astype("float32")  # (S,2) [lon,lat]
T, S = Y.shape

print("Loaded:", BASIN)
print("Y shape (T,S):", Y.shape)
print("Lon range:", float(lons.min()), float(lons.max()))
print("Lat range:", float(lats.min()), float(lats.max()))


Loaded: arabian_gulf
Y shape (T,S): (103792, 332)
Lon range: 48.0 56.75
Lat range: 24.0 30.25


In [7]:
# -----------------------
# Load ENSO / IOD indices
# -----------------------
idx = pd.read_csv(IDX_CSV)

# Parse time (IMPORTANT)
if "time" in idx.columns:
    idx["time"] = pd.to_datetime(idx["time"])
elif {"year", "month"}.issubset(idx.columns):
    idx["time"] = pd.to_datetime(dict(year=idx["year"], month=idx["month"], day=1))
else:
    raise ValueError(
        f"Index CSV needs either 'time' or ('year','month') columns. "
        f"Found: {idx.columns.tolist()}"
    )

# ---- helper to find columns robustly ----
def pick_col(cols, key):
    cols_l = {c.lower(): c for c in cols}
    for cl, orig in cols_l.items():
        if cl == key or key in cl:
            return orig
    return None

roni_col = pick_col(idx.columns, "roni")
dmi_col  = pick_col(idx.columns, "dmi")

if roni_col is None or dmi_col is None:
    raise ValueError(
        f"Could not identify RONI/DMI columns.\n"
        f"Columns found: {idx.columns.tolist()}"
    )

# ---- HARD GUARD / FIX: ensure SST time is monthly month-start ----
t = pd.DatetimeIndex(pd.to_datetime(time))

is_month_start_midnight = (t.day == 1).all() and (t.hour == 0).all() and (t.minute == 0).all()
if not is_month_start_midnight:
    print("⚠️ SST time is not monthly. Converting to monthly month-start via resampling...")
    
    # Rebuild monthly SST + Y + time from your basin dataset
    ds = xr.open_dataset(BASIN_NC)
    da = ds["sst_anom"]
    lat_name = "latitude" if "latitude" in da.coords else "lat"
    lon_name = "longitude" if "longitude" in da.coords else "lon"

    da_m = da.resample(time="MS").mean(skipna=True)
    da_st = da_m.stack(space=(lat_name, lon_name))
    valid_space = np.isfinite(da_st.isel(time=0).values)
    da_st = da_st.isel(space=valid_space)

    Y = da_st.values.astype("float32")
    time = pd.to_datetime(da_st["time"].values)

    ds.close()

    print("✅ Now monthly. Example times:", time[:3])
    print("Y shape (T,S):", Y.shape)
    
# Set time index
idx = idx.set_index("time").sort_index()

# Deduplicate index times if needed
if idx.index.duplicated().any():
    print("⚠️ Duplicate times found in index CSV — deduplicating (keep last).")
    idx = idx[~idx.index.duplicated(keep="last")]

# -----------------------
# Align indices to SST MONTHLY time
# -----------------------
if pd.Index(time).duplicated().any():
    raise ValueError("SST monthly time axis contains duplicates (unexpected).")

idx_aligned = idx.reindex(time)

# Sanity check for missing months
if idx_aligned[[roni_col, dmi_col]].isna().any().any():
    missing = idx_aligned[idx_aligned[roni_col].isna() | idx_aligned[dmi_col].isna()]
    raise ValueError(
        f"Missing index values after aligning to SST months.\n\n"
        f"First missing rows:\n{missing.head()}\n\n"
        f"Index range: {idx.index.min()} → {idx.index.max()}\n"
        f"SST   range: {time.min()} → {time.max()}"
    )

# -----------------------
# Extract standardized predictors
# -----------------------
N = idx_aligned[roni_col].astype("float32").values
D = idx_aligned[dmi_col].astype("float32").values

# z-score (standardized modes)
N = (N - N.mean()) / N.std()
D = (D - D.mean()) / D.std()
ND = (N * D).astype("float32")

print("✅ Index alignment successful.")
print(f"  N (RONI): mean={N.mean():+.3f}, std={N.std():.3f}")
print(f"  D (DMI):  mean={D.mean():+.3f}, std={D.std():.3f}")
print(f"  Time span: {time.min().date()} → {time.max().date()}")
print(f"  Number of months: {len(time)}")
print("Y shape (T,S):", Y.shape)


⚠️ SST time is not monthly. Converting to monthly month-start via resampling...
✅ Now monthly. Example times: DatetimeIndex(['1950-01-01', '1950-02-01', '1950-03-01'], dtype='datetime64[ns]', freq=None)
Y shape (T,S): (912, 332)
⚠️ Duplicate times found in index CSV — deduplicating (keep last).
✅ Index alignment successful.
  N (RONI): mean=+0.000, std=1.000
  D (DMI):  mean=+0.000, std=1.000
  Time span: 1950-01-01 → 2025-12-01
  Number of months: 912
Y shape (T,S): (912, 332)


In [9]:
# X_space is (S,2) with columns [lon, lat] in degrees
lon = X_space[:, 0].astype("float64")
lat = X_space[:, 1].astype("float64")

lat0 = float(lat.mean())
km_per_deg_lat = 111.32
km_per_deg_lon = 111.32 * np.cos(np.deg2rad(lat0))

x_km = (lon - float(lon.mean())) * km_per_deg_lon
y_km = (lat - float(lat.mean())) * km_per_deg_lat

X_km = np.column_stack([x_km, y_km]).astype("float32")

coords = {"time": time, "space": np.arange(S), "xy": ["x_km", "y_km"]}

In [10]:
print("X_km ranges:")
print("  x_km:", float(X_km[:,0].min()), "to", float(X_km[:,0].max()))
print("  y_km:", float(X_km[:,1].min()), "to", float(X_km[:,1].max()))

X_km ranges:
  x_km: -388.7517395019531 to 480.62762451171875
  y_km: -312.33306884765625 to 383.41693115234375


In [11]:
E = z.size
S = Y.shape[1]

coords = {
    "event": np.arange(E),
    "space": np.arange(S),
    "xy": ["x_km", "y_km"],
}

with pm.Model(coords=coords) as m_gpd:
    # Data
    z_obs = pm.ConstantData("z", z, dims="event")
    N_t   = pm.ConstantData("N", N_e, dims="event")
    D_t   = pm.ConstantData("D", D_e, dims="event")
    ND_t  = pm.ConstantData("ND", ND_e, dims="event")
    s_id  = pm.ConstantData("s_id", s_e, dims="event")   # which grid cell each exceedance came from

    # Shared shape (start here; stable)
    xi = pm.Normal("xi", mu=0.05, sigma=0.15)  # mild tails; allows +/-; data will move it
    # Optional: constrain to a plausible range to prevent crazy tails early:
    # xi = pm.TruncatedNormal("xi", mu=0.05, sigma=0.15, lower=-0.3, upper=0.5)

    # Hierarchical priors for cell-wise coefficients on log(sigma)
    a_bar  = pm.Normal("a_bar", 0.0, 1.0)
    bN_bar = pm.Normal("bN_bar", 0.0, 0.5)
    bD_bar = pm.Normal("bD_bar", 0.0, 0.5)
    bND_bar= pm.Normal("bND_bar",0.0, 0.5)

    a_sd   = pm.HalfNormal("a_sd", 0.8)
    bN_sd  = pm.HalfNormal("bN_sd", 0.3)
    bD_sd  = pm.HalfNormal("bD_sd", 0.3)
    bND_sd = pm.HalfNormal("bND_sd",0.2)

    a_s   = pm.Normal("a_s",  mu=a_bar,   sigma=a_sd,   dims="space")
    bN_s  = pm.Normal("bN_s", mu=bN_bar,  sigma=bN_sd,  dims="space")
    bD_s  = pm.Normal("bD_s", mu=bD_bar,  sigma=bD_sd,  dims="space")
    bND_s = pm.Normal("bND_s",mu=bND_bar, sigma=bND_sd, dims="space")

    # log scale for each event, indexed by cell
    log_sigma = (
        a_s[s_id]
        + bN_s[s_id]  * N_t
        + bD_s[s_id]  * D_t
        + bND_s[s_id] * ND_t
    )
    sigma = pm.Deterministic("sigma", pm.math.exp(log_sigma))

    # Likelihood: exceedances z > 0
    pm.GeneralizedPareto("z_like", mu=0.0, sigma=sigma, xi=xi, observed=z_obs)

    idata_gpd = pm.sample(draws=1000, tune=1000, chains=4, target_accept=0.9, random_seed=RANDOM_SEED)




/home/k16v981/.conda/envs/my_env/lib/python3.9/site-packages/pymc/data.py:402: UserWarning: The `mutable` kwarg was not specified. Before v4.1.0 it defaulted to `pm.Data(mutable=True)`, which is equivalent to using `pm.MutableData()`. In v4.1.0 the default changed to `pm.Data(mutable=False)`, equivalent to `pm.ConstantData`. Use `pm.ConstantData`/`pm.MutableData` or pass `pm.Data(..., mutable=False/True)` to avoid this warning.
  warnings.warn(
/home/k16v981/.conda/envs/my_env/lib/python3.9/site-packages/multipledispatch/dispatcher.py:27: AmbiguityWarning: 
Ambiguities exist in dispatched function _unify

The following signatures may result in ambiguous behavior:
	[ConstrainedVar, object, Mapping], [object, ConstrainedVar, Mapping]
	[object, ConstrainedVar, Mapping], [ConstrainedVar, object, Mapping]
	[ConstrainedVar, Var, Mapping], [object, ConstrainedVar, Mapping]
	[ConstrainedVar, Var, Mapping], [object, ConstrainedVar, Mapping]


Consider making the following additions:

@dispatch(

ValueError: Not enough samples to build a trace.

In [ ]:
import arviz as az
az.to_netcdf(idata, OUT_IDATA)
# idata = az.from_netcdf(OUT_IDATA)

# idata.to_netcdf(OUT_IDATA)
# print("Saved idata:", OUT_IDATA)

# # Later, to reload:
# # idata = xr.open_dataset(OUT_IDATA)  # not ideal; use arviz if available

In [ ]:
post = idata.posterior

mu_s  = post["mu_s"].mean(("chain","draw")).values  # (S,)
bN_s  = post["bN_s"].mean(("chain","draw")).values
bD_s  = post["bD_s"].mean(("chain","draw")).values
bND_s = post["bND_s"].mean(("chain","draw")).values

def predict_map_mean(N_star, D_star):
    return mu_s + bN_s*N_star + bD_s*D_star + bND_s*(N_star*D_star)


In [ ]:
def predict_map_ci(N_star, D_star, q=(0.05, 0.95)):
    # use posterior draws
    mu_d  = post["mu_s"].stack(sample=("chain","draw")).values   # (S, samples)
    bN_d  = post["bN_s"].stack(sample=("chain","draw")).values
    bD_d  = post["bD_s"].stack(sample=("chain","draw")).values
    bND_d = post["bND_s"].stack(sample=("chain","draw")).values

    pred = mu_d + bN_d*N_star + bD_d*D_star + bND_d*(N_star*D_star)  # (S, samples)

    lo = np.quantile(pred, q[0], axis=1)
    hi = np.quantile(pred, q[1], axis=1)
    return lo, hi, (hi - lo)


In [ ]:
def plot_map_scatter(values_s, title, vlim=None, cmap=None):
    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=(7.5, 5.5))
    ax = plt.axes(projection=proj)

    ax.coastlines(resolution="110m", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)
    ax.add_feature(cfeature.LAND, linewidth=0.2)
    ax.set_title(title)

    if vlim is None:
        vmax = float(np.nanmax(np.abs(values_s)))
        vmin, vmax = -vmax, vmax
    else:
        vmin, vmax = vlim

    sc = ax.scatter(
        lons, lats, c=values_s,
        s=8, transform=ccrs.PlateCarree(),
        vmin=vmin, vmax=vmax,
        cmap=cmap
    )
    cb = plt.colorbar(sc, ax=ax, shrink=0.8)
    cb.set_label("SST anomaly (K; same as °C anomaly)")
    plt.show()

In [ ]:
# Choose a shared symmetric color scale across experiments
all_preds = []
for ex in EXPERIMENTS:
    all_preds.append(predict_map_mean(ex["N"], ex["D"]))
vmax = float(np.nanmax(np.abs(np.concatenate(all_preds))))
vlim = (-vmax, vmax)

for ex in EXPERIMENTS:
    pred = predict_map_mean(ex["N"], ex["D"])
    plot_map_scatter(pred, f"{BASIN}: predicted SST anomaly — {ex['name']}", vlim=vlim)


In [ ]:
# Choose a shared symmetric color scale across experiments
all_preds = []
for ex in EXPERIMENTS:
    all_preds.append(predict_map_mean(ex["N"], ex["D"]))
vmax = float(np.nanmax(np.abs(np.concatenate(all_preds))))
vlim = (-vmax, vmax)

for ex in EXPERIMENTS:
    pred = predict_map_mean(ex["N"], ex["D"])
    plot_map_scatter(pred, f"{BASIN}: predicted SST anomaly — {ex['name']}", vlim=vlim)


In [ ]:
vmax_b = float(np.nanmax(np.abs(np.concatenate([bN_s, bD_s, bND_s]))))
vlim_b = (-vmax_b, vmax_b)

plot_map_scatter(bN_s,  f"{BASIN}: β_N(s) — response per +1σ RONI", vlim=vlim_b)
plot_map_scatter(bD_s,  f"{BASIN}: β_D(s) — response per +1σ DMI",  vlim=vlim_b)
plot_map_scatter(bND_s, f"{BASIN}: β_ND(s) — interaction term",     vlim=vlim_b)


In [ ]:
ex = {"name": "El Niño (+1,0)", "N": +1.0, "D": 0.0}
lo, hi, width = predict_map_ci(ex["N"], ex["D"], q=(0.05, 0.95))

plot_map_scatter(width, f"{BASIN}: 90% CI width — {ex['name']}", vlim=None)
